# 052 — Set up the MSA-FEMAP695 runs, one per building group

**MSA-FEMAP695** is an MSA with the *record set held fixed*: the same 22 FEMA P695
far-field records are run at every stripe, only the scale factor changes. It is the
third arm of the WP1 fragility comparison —

| arm | protocol | records |
|---|---|---|
| MSA | stripes, binomial MLE fit | 30 GCIM-selected records, **different at every stripe and site** |
| IDA (`ida_femap695`) | hunt-trace-fill, moment fit | 22 FEMA P695 records |
| **MSA-FEMAP695** | stripes, binomial MLE fit | 22 FEMA P695 records, **the same at every stripe** |

Against the site-specific MSA it isolates the effect of the *record set*; against the
IDA it isolates the effect of the *protocol*.

Because the records no longer depend on the site, the analysis depends only on the
structure. Notebook 051 showed the 120 (site, storey) designs collapse to **51 unique
designs** (25 x 3s, 26 x 5s), so these runs are set up **per design group**, not per
site.

## What this notebook writes

1. **Stripe selections** into `results/04_record_selection/AvgSA_03_FEMAP695/` — one
   pickle per (group, stripe), in the same format the site-specific selections use so
   `standes.analysis.msa` reads them unchanged.
2. **Analysis folders** under `DEST_ROOT/group_{n}s_{gid}/{n}s/mdof/`
   (`DEST_ROOT = analysis_data.wp1_fixed_record_sets`, i.e. `D:/08_wp1_fixed_record_sets`).
   The `{n}s` level is redundant with the group name but keeps the folder depth
   identical to the site-specific `site_{ii}/{n}s/mdof/` tree, so the same launchers,
   `window_name_index` and post-processing paths work for both.
   Each folder gets the structural model, a modal analysis and the MSA files —
   no cyclic pushover, no IDA.
3. **Batch launchers** at
   `phd_project/scripts/WP1_ground_motion_set/batch_run_analyses/fixed_record_sets/{n}s/mdof/msa_AvgSA03_femap695.py`.

`DEST_ROOT` will later hold other fixed record sets alongside this one — hence the
record set in the config filename (`config_msa_AvgSA_03_femap695.py`), the stripe
folder (`AvgSA_03_femap695/`) and the results folder (`msa_AvgSA_03_femap695/`).

## Choosing the stripes

There is no hazard curve here — the stripes are chosen so they straddle the collapse
fragility. Each group's **representative structure's FEMA P695 IDA fragility**
(`{rep_tag}_ida_femap695_collapsefragility_AvgSA_03.json`, nb 014) is inverted at
**P[C] = 0.15, 0.40, 0.60, 0.85**:

$$\mathrm{IML}(p) = \theta \exp\!\big(\beta\,\Phi^{-1}(p)\big)$$

A group whose representative has **no IDA fragility yet is skipped with a printed
note** — today that is every 5s group, since the 5s IDAs have not been run. Re-run this
notebook once they land.

`rtp` and `poe` are stored as `NaN`: the stripes are defined by collapse probability,
not by hazard, so a return period would be meaningless. The intended P[C] is kept in a
`target_poc` key, which the MSA runner ignores.

## Dependencies

- **011** — the per-site designs (`{tag}_out.json`) the group models are copied from.
- **014** — the converted `ida_femap695` AvgSA_03 fragilities that set the stripes.
- **051** — `unique_structural_designs.csv`, the design groups.

## Running this notebook without the analysis drive

Set `BUILD_ANALYSIS_FOLDERS = False` to write only the in-repo artefacts (stripe
selections, launchers, group map) and skip creating the folders on `D:`. Everything
else, including the paths written into the launchers, is unaffected.

In [1]:
# %load_ext autoreload
# %autoreload 2

## 0. Setup & parameters

In [2]:
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import lognorm

from phd_project.config import config
from phd_project.scripts.cache_utils import fingerprint, write_manifest
from phd_project.scripts.case_study_design_scripts.design_file_helpers import (
    get_n_damping_modes_from_design_file,
    get_n_primary_modes_from_design_file,
)
from phd_project.scripts.templates.copy_templates_to_folders import (
    copy_analysis_config,
    copy_batch_ida_buildings,
    copy_file,
    copy_nlcbf_model,
)
from phd_project.scripts.WP1_ground_motion_set.design_groups import load_design_groups
from phd_project.scripts.WP1_ground_motion_set.gm_selection import iml_filename_tag
from standes.groundmotion import load_ground_motion_from_json
from standes.intensitymeasures import avgsa_03

cfg = config.load_config()

C:\Users\clemettn\Documents\phd\phd_project\scripts\WP1_ground_motion_set\gm_selection.py:15: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [3]:
# ----------------------------------------------------------------------------
# PARAMETERS
# ----------------------------------------------------------------------------
# Root of the (large) analysis folders. Written as DEST_ROOT/group_{n}s_{gid}/{n}s/mdof/.
DEST_ROOT = cfg["analysis_data"]["wp1_fixed_record_sets"]

# The DEST_ROOT folders can only be written where the analysis drive is attached.
# False (as last run here) writes just the in-repo artefacts - stripe selections,
# launchers and the group map - and the launcher paths are the DEST_ROOT ones either
# way. SET THIS TO True ON THE ANALYSIS MACHINE to actually build the run folders.
BUILD_ANALYSIS_FOLDERS = False

# --- record set / intensity measure ---
GM_SET = "AvgSA_03"                 # the conditioning IM of the stripes
RECORD_SET = "femap695"             # names the config / stripe folder / results folder
COND_IMT = "AvgSA([0, 3])"          # as written in the site-specific stripe pickles
IDA_TAG = "ida_femap695"            # the IDA whose fragility sets the stripes

# Records as seen BY THE ANALYSIS MACHINE (the MSA worker resolves
# record_src / filename). Same folder nb 011's IDA reads.
GM_JSON_SRC = "D:/gm_records_p695"
# The identical records in-repo, used HERE to compute each record's AvgSA_03 so the
# notebook runs with no external drive attached.
LOCAL_RECORD_SRC = Path(cfg["raw_data"]["femap695_records"])

# --- stripes ---
STOREYS = [3, 5]
TARGET_POCS = [0.15, 0.40, 0.60, 0.85]   # P[collapse] each stripe should land at
IML_DECIMALS = 3                         # resolution encoded by iml_filename_tag

# --- structural model (matches nb 011) ---
DAMPING_RATIO = 0.05
MDOF_DRIFT_LIMIT = 0.2                   # MDOF collapse drift limit
RECORDER_KEY = "ida_process_recorder_roof&maxstorey_drift"

# --- MSA ---
STRIPE_ORDER_ASCENDING = True
MAX_N_RECORDS = None                     # cap records per stripe (None = all 22)

# --- paths ---
DESIGN_ROOT = Path(cfg["models"]["casestudy_designs_site_specific"])
FRAG_ROOT = Path(cfg["proc_data"]["wp1_sites_fragility_curves"])
SELECTION_DST = Path(cfg["results"][f"{GM_SET}_FEMAP695_record_selection"])
SELECTION_DST.mkdir(parents=True, exist_ok=True)
BATCH_BASE = Path(cfg["scripts"]["batch_run_analyses"]) / "fixed_record_sets"

# derived names
STRIPE_DIRNAME = f"{GM_SET}_{RECORD_SET}"             # AvgSA_03_femap695
CONFIG_NAME = f"config_msa_{GM_SET}_{RECORD_SET}.py"  # config_msa_AvgSA_03_femap695.py
RESULTS_FOLDER_NAME = f"msa_{GM_SET}_{RECORD_SET}"    # msa_AvgSA_03_femap695

print(f"DEST_ROOT     = {DEST_ROOT}  (build folders: {BUILD_ANALYSIS_FOLDERS})")
print(f"SELECTION_DST = {SELECTION_DST}")
print(f"BATCH_BASE    = {BATCH_BASE}")

DEST_ROOT     = D:\08_wp1_fixed_record_sets  (build folders: False)
SELECTION_DST = C:\Users\clemettn\Documents\phd\results\04_record_selection\AvgSA_03_FEMAP695
BATCH_BASE    = C:\Users\clemettn\Documents\phd\phd_project\scripts\WP1_ground_motion_set\batch_run_analyses\fixed_record_sets


## 1. The building groups

One row per design group from notebook 051 (`is_representative == True`). `group_id`
restarts at 0 for each storey count, so the folder name carries both:
`group_{n}s_{gid:02d}`.

In [4]:
groups_df = load_design_groups(cfg)

groups = (groups_df[groups_df["is_representative"] & groups_df["storeys"].isin(STOREYS)]
          .loc[:, ["storeys", "group_id", "representative_site",
                   "representative_tag", "n_sites_in_group"]]
          .sort_values(["storeys", "group_id"])
          .reset_index(drop=True))

# member sites of each group, for the map written in section 7
members = (groups_df.groupby(["storeys", "group_id"])["site"]
           .apply(lambda s: sorted(int(x) for x in s)))


def group_folder_name(n: int, gid: int) -> str:
    """Folder for a design group: group_3s_00, group_5s_07, ..."""
    return f"group_{n}s_{int(gid):02d}"


def mdof_folder(n: int, gid: int) -> Path:
    """DEST_ROOT/group_{n}s_{gid}/{n}s/mdof - the {n}s level is deliberately
    redundant, so the folder depth matches the site-specific site_{ii}/{n}s/mdof tree."""
    return DEST_ROOT / group_folder_name(n, gid) / f"{n}s" / "mdof"


for n in STOREYS:
    n_groups = int((groups["storeys"] == n).sum())
    n_sites = int(groups_df[groups_df["storeys"] == n]["site"].nunique())
    print(f"{n}s: {n_groups} design groups covering {n_sites} sites")
groups.head()

3s: 25 design groups covering 60 sites
5s: 26 design groups covering 60 sites


,storeys,group_id,representative_site,representative_tag,n_sites_in_group
0,3,0,0,3s_cbf_dc2_site0,8
1,3,1,1,3s_cbf_dc2_site1,6
2,3,2,3,3s_cbf_dc2_site3,1
3,3,3,7,3s_cbf_dc2_site7,5
4,3,4,9,3s_cbf_dc2_site9,3


## 2. Stripe IMLs from each group's representative IDA fragility

The fragility is a lognormal in AvgSA_03 [g] fitted by notebook 014 from the
representative structure's FEMA P695 IDA. Inverting it at the four target collapse
probabilities gives the stripe IMLs.

A group with no fragility on disk is **reported and dropped** — no error. All 5s groups
fall in that category until the 5s IDAs have been run.

In [5]:
def fragility_path(rep_site: int, rep_tag: str) -> Path:
    return (FRAG_ROOT / f"site_{int(rep_site)}"
            / f"{rep_tag}_{IDA_TAG}_collapsefragility_{GM_SET}.json")


def stripe_imls(median: float, dispersion: float) -> list[float]:
    """The IMLs [g] at which the fragility reaches each target P[collapse].

    P[C|im] = lognorm.cdf(im, s=dispersion, scale=median), so the inverse is
    lognorm.ppf. Rounded to the resolution the stripe filenames encode; a duplicate
    after rounding would collide on filename, so duplicates are dropped."""
    imls = lognorm.ppf(TARGET_POCS, dispersion, scale=median)
    return sorted({round(float(x), IML_DECIMALS) for x in imls})


stripes: dict[tuple[int, int], dict] = {}   # (storeys, group_id) -> group info
skipped: list[tuple[int, int]] = []

for row in groups.itertuples(index=False):
    n, gid = int(row.storeys), int(row.group_id)
    fc_path = fragility_path(row.representative_site, row.representative_tag)
    if not fc_path.is_file():
        print(f"[skip] group {n}s/{gid:02d} ({row.representative_tag}): "
              f"no {IDA_TAG} fragility at {fc_path}")
        skipped.append((n, gid))
        continue

    fc = json.loads(fc_path.read_text())
    imls = stripe_imls(fc["median"], fc["dispersion"])
    stripes[(n, gid)] = {
        "storeys": n,
        "group_id": gid,
        "representative_site": int(row.representative_site),
        "representative_tag": row.representative_tag,
        "n_sites": int(row.n_sites_in_group),
        "fragility_path": fc_path,
        "median": float(fc["median"]),
        "dispersion": float(fc["dispersion"]),
        "imls": imls,
    }

print(f"\n{len(stripes)} group(s) with stripes, {len(skipped)} skipped")
summary = pd.DataFrame([
    {"group": group_folder_name(g["storeys"], g["group_id"]),
     "rep_tag": g["representative_tag"], "n_sites": g["n_sites"],
     "median [g]": g["median"], "beta": g["dispersion"],
     **{f"P[C]={p:.2f}": iml for p, iml in zip(TARGET_POCS, g["imls"])}}
    for g in stripes.values()])
with pd.option_context("display.float_format", lambda v: f"{v:.3f}"):
    print(summary.to_string(index=False))

[skip] group 5s/00 (5s_cbf_dc2_site0): no ida_femap695 fragility at C:\Users\clemettn\Documents\phd\data_processed\09_structure_fragility_curves\wp1_casestudy_sites\site_0\5s_cbf_dc2_site0_ida_femap695_collapsefragility_AvgSA_03.json
[skip] group 5s/01 (5s_cbf_dc2_site7): no ida_femap695 fragility at C:\Users\clemettn\Documents\phd\data_processed\09_structure_fragility_curves\wp1_casestudy_sites\site_7\5s_cbf_dc2_site7_ida_femap695_collapsefragility_AvgSA_03.json
[skip] group 5s/02 (5s_cbf_dc2_site10): no ida_femap695 fragility at C:\Users\clemettn\Documents\phd\data_processed\09_structure_fragility_curves\wp1_casestudy_sites\site_10\5s_cbf_dc2_site10_ida_femap695_collapsefragility_AvgSA_03.json
[skip] group 5s/03 (5s_cbf_dc2_site11): no ida_femap695 fragility at C:\Users\clemettn\Documents\phd\data_processed\09_structure_fragility_curves\wp1_casestudy_sites\site_11\5s_cbf_dc2_site11_ida_femap695_collapsefragility_AvgSA_03.json
[skip] group 5s/04 (5s_cbf_dc2_site17): no ida_femap695 fr

## 3. FEMA P695 record intensities

Every stripe of every group uses the same 22 records, so each record's *unscaled*
AvgSA_03 is computed once here and reused throughout. `gravity_factor=1.0` keeps the
result in **g** — the record JSONs store accelerations in g, and the fragility medians
are in g — so `alpha = target_iml / record_iml` is a clean dimensionless ratio.

In [6]:
FEMAP695_RECORDS = sorted(p.name for p in LOCAL_RECORD_SRC.glob("fema_p695_*.json"))
if not FEMAP695_RECORDS:
    raise FileNotFoundError(f"no fema_p695_*.json under {LOCAL_RECORD_SRC}")

im = avgsa_03()
record_im: dict[str, float] = {}
for name in FEMAP695_RECORDS:
    gm = load_ground_motion_from_json(LOCAL_RECORD_SRC / name)
    im.set_iml(gm, 1.0)          # gravity_factor = 1 -> IML in g
    record_im[name] = float(im.iml)

record_im_arr = np.array([record_im[n] for n in FEMAP695_RECORDS])
print(f"{len(FEMAP695_RECORDS)} FEMA P695 records from {LOCAL_RECORD_SRC}")
print(f"unscaled {GM_SET} [g]: min {record_im_arr.min():.4f}, "
      f"median {np.median(record_im_arr):.4f}, max {record_im_arr.max():.4f}")

22 FEMA P695 records from C:\Users\clemettn\Documents\phd\data_raw\fema_P695_records
unscaled AvgSA_03 [g]: min 0.1083, median 0.2330, max 0.3419


## 4. Build and save the stripe selections

Each stripe is written in the format `standes.analysis.msa.read_stripe_selection`
expects, so no runner code changes: a dict with `cond_imt`, `cond_iml`, `rtp`, `poe`
and a `recs` dataframe whose columns are a `("metadata", ...)` MultiIndex. Only
`("metadata","filename")` and `("metadata","alpha")` are read — filename resolves the
record against `record_src`, alpha is the scale factor applied verbatim — so those are
the only two columns written. Row order fixes the `record_{k}` tags.

The filename must contain `stripe` and `gm_selection` (so `find_stripe_pickles` picks
it up) and the `stripe_iml_NNptNNN` token (so stripes order numerically and the result
folders are intensity-keyed).

In [7]:
def build_stripe_selection(iml: float, target_poc: float) -> dict:
    """One stripe: the 22 records, each scaled to reach `iml` in AvgSA_03."""
    recs = pd.DataFrame({
        "filename": FEMAP695_RECORDS,
        "alpha": [iml / record_im[name] for name in FEMAP695_RECORDS],
    })
    recs.columns = pd.MultiIndex.from_product([["metadata"], recs.columns])
    return {
        "cond_imt": COND_IMT,
        "cond_iml": float(iml),
        "rtp": float("nan"),        # no hazard behind these stripes
        "poe": float("nan"),        # ditto - the design intent is target_poc
        "target_poc": float(target_poc),
        "recs": recs,
    }


def selection_path(n: int, gid: int, iml: float) -> Path:
    return (SELECTION_DST / f"{group_folder_name(n, gid)}"
            f"__stripe_iml_{iml_filename_tag(iml)}__gm_selection.pickle")


written: list[Path] = []
for (n, gid), g in stripes.items():
    paths = []
    for iml, poc in zip(g["imls"], TARGET_POCS):
        selection = build_stripe_selection(iml, poc)
        fp = selection_path(n, gid, iml)
        with open(fp, "wb") as file:
            pickle.dump(selection, file)
        write_manifest(fp, fingerprint(
            fragility=g["fragility_path"],
            record_names=tuple(FEMAP695_RECORDS),
            record_ims=record_im_arr,
            target_poc=float(poc),
            cond_iml=float(iml),
        ))
        paths.append(fp)
        written.append(fp)
    g["selection_paths"] = paths

print(f"wrote {len(written)} stripe selection(s) to {SELECTION_DST}")
if written:
    example = pickle.load(open(written[0], "rb"))
    print(f"\nexample {written[0].name}: cond_iml = {example['cond_iml']} g, "
          f"{len(example['recs'])} records")
    print(example["recs"].head(3).to_string())

wrote 100 stripe selection(s) to C:\Users\clemettn\Documents\phd\results\04_record_selection\AvgSA_03_FEMAP695

example group_3s_00__stripe_iml_00pt314__gm_selection.pickle: cond_iml = 0.314 g, 22 records
                metadata          
                filename     alpha
0  fema_p695_120111.json  1.292003
1  fema_p695_120121.json  1.141618
2  fema_p695_120411.json  1.190020


## 5. Build the group analysis folders

Each group's model is the design of its representative site, copied from
`casestudy_designs_site_specific`. As in notebook 011 two model variants are written:
`initialise_model.py` (full recorders, used by the modal analysis) and
`initialise_model_idamsa.py` (reduced recorders, used by the MSA).

The stripe folder is cleared of any `*__stripe_*__gm_selection.pickle` before the
current run list is copied in, so a re-run with different target P[C] cannot leave a
stale stripe behind for `find_stripe_pickles` to sweep up.

In [8]:
def build_group_folder(g: dict) -> Path:
    """Write one group's mdof folder: model + modal + MSA-FEMAP695. Returns the folder."""
    n, gid, tag = g["storeys"], g["group_id"], g["representative_tag"]
    folder = mdof_folder(n, gid)
    folder.mkdir(parents=True, exist_ok=True)

    # --- design file (the model reads it from its own folder) ---
    design_src = DESIGN_ROOT / tag / f"{tag}_out.json"
    design_dst = folder / f"{tag}_designfile.json"
    copy_file(design_src, design_dst)

    # --- structural model, full recorders (modal) and reduced recorders (MSA) ---
    model_kwargs = dict(
        design_json=design_dst.name,
        damping_updates={"n_modes": get_n_damping_modes_from_design_file(design_dst),
                         "damping_ratio": DAMPING_RATIO},
        recorder_updates={"drift_limit": MDOF_DRIFT_LIMIT},
    )
    init_fn_full = copy_nlcbf_model(cfg["templates"], folder, **model_kwargs)
    init_fn_reduced = copy_nlcbf_model(cfg["templates"], folder, reduced=True, **model_kwargs)

    # --- modal analysis (fundamental period) ---
    copy_file(cfg["templates"]["run_modal"], folder / "run_modal.py")
    copy_analysis_config(
        cfg["templates"]["config_modal"],
        folder / "config_modal.py",
        results_folder_name="modal",
        model_file_name=init_fn_full,
        update_config={"n_modes": get_n_primary_modes_from_design_file(design_dst)},
    )

    # --- MSA coordinator / worker / helpers ---
    copy_file(cfg["templates"]["run_batch_msa_per_record"],
              folder / "run_batch_msa_per_record.py")
    copy_file(cfg["templates"]["run_msa_per_record"], folder / "run_msa_per_record.py")
    copy_file(cfg["templates"]["nltha_injection_update_damping"],
              folder / "injection_functions.py")
    copy_file(cfg["templates"][RECORDER_KEY], folder / "msa_process_recorders.py")

    # --- stripe pickles ---
    stripe_dir = folder / STRIPE_DIRNAME
    stripe_dir.mkdir(parents=True, exist_ok=True)
    for old in stripe_dir.glob("*__stripe_*__gm_selection.pickle"):
        old.unlink()
    for src in g["selection_paths"]:
        copy_file(src, stripe_dir / src.name)

    # --- MSA config. record_src is the P695 folder, not data_processed/07_gm_records:
    # the worker resolves each record as record_src / ("metadata","filename").
    copy_analysis_config(
        cfg["templates"]["config_msa"],
        folder / CONFIG_NAME,
        results_folder_name=RESULTS_FOLDER_NAME,
        model_file_name=init_fn_reduced,
        gm_selection_src_str=stripe_dir.as_posix(),
        record_src_str=GM_JSON_SRC,
        stripe_order_ascending=STRIPE_ORDER_ASCENDING,
        max_n_records=MAX_N_RECORDS,
    )
    return folder

In [9]:
built: dict[int, list[dict]] = {n: [] for n in STOREYS}

for (n, gid), g in sorted(stripes.items()):
    folder = mdof_folder(n, gid)
    if BUILD_ANALYSIS_FOLDERS:
        design_src = DESIGN_ROOT / g["representative_tag"] / f"{g['representative_tag']}_out.json"
        if not design_src.is_file():
            print(f"WARNING: skipping group {n}s/{gid:02d}: {design_src} not found "
                  f"(run nb 011 first)")
            continue
        folder = build_group_folder(g)
    built[n].append({**g, "folder": folder})

for n in STOREYS:
    verb = "built" if BUILD_ANALYSIS_FOLDERS else "listed (folders NOT built)"
    print(f"{n}s: {len(built[n])} group folder(s) {verb}")

3s: 25 group folder(s) listed (folders NOT built)
5s: 0 group folder(s) listed (folders NOT built)


## 6. Batch launchers

One `run_batch_msa_buildings` launcher per storey, listing every group of that storey.
`window_name_index=2` puts the `group_{n}s_{gid}` folder in the console title (the
config sits at `group_.../{n}s/mdof/`, so two levels up is the group).

The paths written are always the `DEST_ROOT` ones, whether or not the folders were
created here — the launcher is run on the analysis machine.

In [10]:
batch_name = f"msa_{GM_SET.replace('_', '')}_{RECORD_SET}.py"   # msa_AvgSA03_femap695.py

for n in STOREYS:
    if not built[n]:
        print(f"{n}s: no groups - no launcher written")
        continue
    buildings = [{"folder": g["folder"], "config": g["folder"] / CONFIG_NAME}
                 for g in built[n]]
    batch_dir = BATCH_BASE / f"{n}s" / "mdof"
    batch_dir.mkdir(parents=True, exist_ok=True)
    batch_dst = batch_dir / batch_name
    copy_batch_ida_buildings(
        cfg["templates"]["run_batch_msa_buildings"], batch_dst, buildings,
        window_name_index=2)
    print(f"wrote {batch_dst.relative_to(BATCH_BASE)}: {len(buildings)} buildings")

wrote 3s\mdof\msa_AvgSA03_femap695.py: 25 buildings
5s: no groups - no launcher written


## 7. Group -> site map

One row per group, listing the member sites its MSA-FEMAP695 result stands for, the
stripe IMLs and the P[C] each was aimed at. Notebooks 060 / 061 use this to fan a
group's fragility back out to its sites.

In [11]:
rows = []
for n in STOREYS:
    for g in built[n]:
        sites = members[(n, g["group_id"])]
        rows.append({
            "storeys": n,
            "group_id": g["group_id"],
            "folder": Path(g["folder"]).as_posix(),
            "representative_site": g["representative_site"],
            "representative_tag": g["representative_tag"],
            "n_sites": len(sites),
            "sites": " ".join(str(s) for s in sites),
            "median": g["median"],
            "dispersion": g["dispersion"],
            "imls": " ".join(f"{x:.3f}" for x in g["imls"]),
            "target_pocs": " ".join(f"{p:.2f}" for p in TARGET_POCS),
        })

group_map = pd.DataFrame(rows)
map_path = SELECTION_DST / "group_folder_map.csv"
group_map.to_csv(map_path, index=False)
print(f"wrote {map_path}  ({len(group_map)} groups)")
group_map.head()

wrote C:\Users\clemettn\Documents\phd\results\04_record_selection\AvgSA_03_FEMAP695\group_folder_map.csv  (25 groups)


,storeys,group_id,folder,representative_site,representative_tag,n_sites,sites,median,dispersion,imls,target_pocs
0,3,0,D:/08_wp1_fixed_record_sets/group_3s_00/3s/mdof,0,3s_cbf_dc2_site0,8,0 4 8 19 25 26 27 29,0.393142,0.216237,0.314 0.372 0.415 0.492,0.15 0.40 0.60 0.85
1,3,1,D:/08_wp1_fixed_record_sets/group_3s_01/3s/mdof,1,3s_cbf_dc2_site1,6,1 2 5 6 16 23,0.382000,0.190402,0.314 0.364 0.401 0.465,0.15 0.40 0.60 0.85
2,3,2,D:/08_wp1_fixed_record_sets/group_3s_02/3s/mdof,3,3s_cbf_dc2_site3,1,3,0.382527,0.188868,0.315 0.365 0.401 0.465,0.15 0.40 0.60 0.85
3,3,3,D:/08_wp1_fixed_record_sets/group_3s_03/3s/mdof,7,3s_cbf_dc2_site7,5,7 10 11 18 24,0.466230,0.190812,0.383 0.444 0.489 0.568,0.15 0.40 0.60 0.85
4,3,4,D:/08_wp1_fixed_record_sets/group_3s_04/3s/mdof,9,3s_cbf_dc2_site9,3,9 20 22,0.360199,0.182117,0.298 0.344 0.377 0.435,0.15 0.40 0.60 0.85
